# Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

- 🤝 Breakout Room #1
  1. Task 1: Installing Required Libraries
  2. Task 2: Set Environment Variables
  3. Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  4. Task 4: Evaluating our Pipeline with Ragas
  5. Task 6: Making Adjustments and Re-Evaluating

But first! Let's set some dependencies!

## Dependencies and API Keys:

We'll also need to provide our API keys.

First, OpenAI's for our LLM/embedding model combination!

In [1]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

c:\AIProjects\bootcamp\AIE7\08_Evaluating_RAG_With_Ragas\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
c:\AIProjects\bootcamp\AIE7\08_Evaluating_RAG_With_Ragas\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
c:\AIProjects\bootcamp\AIE7\08_Evaluating_RAG_With_Ragas\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [4]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '5fdc13'. Skipping!
Property 'summary' already exists in node 'dc47eb'. Skipping!
Property 'summary' already exists in node '73f4e3'. Skipping!
Property 'summary' already exists in node '6e881c'. Skipping!
Property 'summary' already exists in node '17f119'. Skipping!
Property 'summary' already exists in node '4ee57f'. Skipping!
Property 'summary' already exists in node 'eaea74'. Skipping!
Property 'summary' already exists in node '32e548'. Skipping!
Property 'summary' already exists in node '6e01de'. Skipping!
Property 'summary' already exists in node 'ef7581'. Skipping!
Property 'summary' already exists in node '0b0040'. Skipping!
Property 'summary' already exists in node '28c954'. Skipping!
Property 'summary' already exists in node 'd55a9d'. Skipping!
Property 'summary' already exists in node 'e896d6'. Skipping!


...

Property 'summary_embedding' already exists in node '73f4e3'. Skipping!
Property 'summary_embedding' already exists in node 'eaea74'. Skipping!
Property 'summary_embedding' already exists in node '6e881c'. Skipping!
Property 'summary_embedding' already exists in node '5fdc13'. Skipping!
Property 'summary_embedding' already exists in node '4ee57f'. Skipping!
Property 'summary_embedding' already exists in node 'e896d6'. Skipping!
Property 'summary_embedding' already exists in node '17f119'. Skipping!
Property 'summary_embedding' already exists in node 'dc47eb'. Skipping!
Property 'summary_embedding' already exists in node '32e548'. Skipping!
Property 'summary_embedding' already exists in node '0b0040'. Skipping!
Property 'summary_embedding' already exists in node '28c954'. Skipping!
Property 'summary_embedding' already exists in node '6e01de'. Skipping!
Property 'summary_embedding' already exists in node 'ef7581'. Skipping!
Property 'summary_embedding' already exists in node 'd55a9d'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [5]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"What guidance does Volume 7, Chapter 4 provide...","[non-term (includes clock-hour calendars), or ...","Volume 7, Chapter 4 provides information on th...",single_hop_specifc_query_synthesizer
1,Are physical therapy clinical experiences that...,[Inclusion of Clinical Work in a Standard Term...,If the clinical work in a physical therapy pro...,single_hop_specifc_query_synthesizer
2,FWS gotta do with payment period?,[Non-Term Characteristics A program that measu...,Payment period is for all Title IV programs ex...,single_hop_specifc_query_synthesizer
3,Whaat is the efffect of a studdent's accelerra...,[both the credit or clock hours and the weeks ...,The Pell Grant or TEACH Grant amount that a st...,single_hop_specifc_query_synthesizer
4,If a medical or education program requires a p...,[<1-hop>\n\nInclusion of Clinical Work in a St...,When a medical or education program requires a...,multi_hop_abstract_query_synthesizer
5,When a practicum or clinical experience is req...,[<1-hop>\n\nInclusion of Clinical Work in a St...,A practicum or clinical experience required fo...,multi_hop_abstract_query_synthesizer
6,if a student in a medical program gotta do a p...,[<1-hop>\n\nInclusion of Clinical Work in a St...,if all students in a medical program gotta do ...,multi_hop_abstract_query_synthesizer
7,if practicum or clinical experience is require...,[<1-hop>\n\nInclusion of Clinical Work in a St...,if all students must do practicum or clinical ...,multi_hop_abstract_query_synthesizer
8,What are the disbursement requirements for Tit...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,"In subscription-based programs, for the first ...",multi_hop_specific_query_synthesizer
9,How do the characteristics of non-term and sub...,[<1-hop>\n\nnon-term (includes clock-hour cale...,The characteristics of non-term and subscripti...,multi_hop_specific_query_synthesizer


## LangChain RAG

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [6]:
path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

1102

#### ❓ Question: 

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### ✅ Answer

The chunk_overlap parameter in the RecursiveCharacterTextSplitter determines how many characters from the end of one chunk are repeated at the start of the next chunk when splitting text. This overlap helps preserve important context that might otherwise be lost at chunk boundaries, ensuring that information spanning two chunks is not missed. By maintaining this continuity, retrieval and generation tasks—such as those in RAG systems—can produce more accurate and coherent results. Overall, chunk overlap reduces information loss and improves the effectiveness of document processing pipelines.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [9]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="loan_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="loan_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [10]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

Now we can produce a node for retrieval!

In [12]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### Augmented

Let's create a simple RAG prompt!

In [13]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [15]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [17]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [18]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [19]:
response = graph.invoke({"question" : "What are the different kinds of loans?"})

In [20]:
response["response"]

'Based on the provided context, the different kinds of loans mentioned are:\n\n1. **Direct Loan** (including both subsidized and unsubsidized types)\n2. **Direct Unsubsidized Loan** (a subtype of Direct Loan)\n\nThe context specifically discusses the options and considerations related to these loans, such as interest accrual, repayment plans, and borrower obligations.'

## Evaluating the App with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [21]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [22]:
dataset.samples[0].eval_sample.response

'Volume 7, Chapter 4 provides guidance that for subscription-based academic calendars, the usual methods of calculating Pell Grant awards apply similarly to other program structures. Specifically, it states that for programs with traditional credit-hour offerings and scheduled to be completed within a specific time frame, the program can generally be considered term-based. \n\nHowever, the chapter emphasizes that if the program does not conform to a traditional academic calendar or meet certain other criteria, additional considerations may be necessary. While the chapter provides an example (Example 6) for standard terms of equal length and less than 30 weeks, it notes that for subscription-based programs (see Volume 3, Chapter 1), special guidance exists. \n\nIn summary, Volume 7, Chapter 4 indicates that the calculation process involves prorating the Scheduled Award based on the fraction of the instructional time in the payment period relative to the full academic year, using appropr

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [23]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [24]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [25]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[59]: TimeoutError()
Exception raised in Job[71]: TimeoutError()


{'context_recall': 0.8194, 'faithfulness': 0.8776, 'factual_correctness': 0.6017, 'answer_relevancy': 0.8712, 'context_entity_recall': 0.3069, 'noise_sensitivity_relevant': 0.1643}

## Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!

> NOTE: This will be using Cohere's Rerank model - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

In [26]:
os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")


We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [27]:
adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [28]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [29]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [30]:
response = adjusted_graph.invoke({"question" : "What are the different kinds of loans?"})
response["response"]

'The provided context mentions two specific types of loans: \n\n1. **Federal PLUS Loans**, which include **Direct PLUS Loans** and **Federal PLUS Loans made under the FFEL Program** before July 1, 2010.\n2. **Direct Subsidized Loans** and **Direct Unsubsidized Loans**, which are available to undergraduate and graduate or professional students, respectively.\n\nWhile the context primarily discusses these loan types and their limits, the explicit mention of different kinds of loans includes:\n\n- **Federal PLUS Loans** (including Direct PLUS Loans and Federal PLUS Loans under FFEL)\n- **Direct Subsidized Loans**\n- **Direct Unsubsidized Loans**\n\nThese are the different kinds of loans referenced in the provided information.'

In [31]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

In [32]:
rerank_dataset.samples[0].eval_sample.response

'Volume 7, Chapter 4 provides guidance that for subscription-based academic calendars, when the instructional weeks in the fall and spring terms are less than 30 weeks, the full-time Scheduled Pell Grant award must be prorated based on the actual number of weeks in the academic calendar. Specifically, the annual Pell Grant award is multiplied by the ratio of the total weeks in the academic year to 30 weeks (the minimum instructional time for a full academic year). The resulting prorated amount is then divided by the number of terms in the fall through spring to determine the award for each term.'

In [33]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [34]:
result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

{'context_recall': 0.7801, 'faithfulness': 0.8431, 'factual_correctness': 0.5933, 'answer_relevancy': 0.8491, 'context_entity_recall': 0.3199, 'noise_sensitivity_relevant': 0.1901}

Original:
'context_recall': 0.8194, 
'faithfulness': 0.8776, 
'factual_correctness': 0.6017, 
'answer_relevancy': 0.8712,
'context_entity_recall': 0.3069, 'noise_sensitivity_relevant': 0.1643

Cohere/Rerank:
{'context_recall': 0.7801, 
'faithfulness': 0.8431, 
'factual_correctness': 0.5933, 
'answer_relevancy': 0.8491, 
'context_entity_recall': 0.3199, 'noise_sensitivity_relevant': 0.1901}

#### ❓ Question: 

Which system performed better, on what metrics, and why?

##### ✅ Answer

The Cohere reranking system underperformed the original system across most metrics, with lower scores in context recall (0.7801 vs 0.8194), faithfulness (0.8431 vs 0.8776), factual correctness (0.5933 vs 0.6017), and answer relevancy (0.8491 vs 0.8712). Theoretically, Cohere reranking should have been better because it uses a more sophisticated cross-encoder model that considers the relationship between the query and each document simultaneously, rather than just comparing embeddings independently. This approach typically provides more accurate relevance scoring by understanding semantic relationships and context better than simple vector similarity. However, this unexpected result likely occurred because the reranking process may have been too aggressive in filtering out relevant context, or the specific test dataset generated by Ragas contained queries that didn't benefit from the reranking approach. Additionally, the reranking model might not have been optimally tuned for this particular domain of financial loan documents, and the evaluation dataset size (only 10 test samples) may have been too small to capture the true performance differences between the two approaches.